# Codestral Mamba 7B (FP4-KM + LoRA) trên Colab T4 bằng Vulkan

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từ trên xuống. Notebook cài Nim 2.2.10 và Vulkan, clone repo kèm submodule, tải `codestral_mamba_only.nimq`, tokenizer và adapter LoRA từ Hugging Face, build `infer_lora` và `formers-cli`, rồi chạy inference trên Vulkan. Tokenizer chạy thuần Nim.

LoRA luôn được merge vào weight FP4 ngay trên GPU, nên decode không còn phép cộng adapter. Prefill dùng Tensor Core của T4 (`KHR_cooperative_matrix`, FP16 nhân, FP32 cộng dồn). FP4 chỉ là định dạng lưu trữ. Decode là GEMV FP4, được autotune theo từng shape.

Sửa `PROMPT`, `MAX_NEW_TOKENS` ở cell inference.

In [1]:
# 1) Kiểm tra runtime GPU và cài các gói hệ thống, gồm Vulkan SDK/runtime và shader compiler.
import os, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), 'Notebook này cần chạy trên Google Colab.'
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True))
except Exception as e:
    raise RuntimeError('Chưa có GPU NVIDIA. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.') from e

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','xz-utils','pkg-config','time','libvulkan1','libvulkan-dev','vulkan-tools','glslc','libegl1','libglx0','libx11-6','libxext6','libx11-xcb1','libxcb1','libxcb-dri3-0','libxcb-present0','libxfixes3','libxrandr2','libdrm2','libexpat1','libwayland-client0'], check=True)


Tesla T4, 580.82.07, 15360 MiB



CompletedProcess(args=['apt-get', 'install', '-y', '-qq', 'build-essential', 'git', 'curl', 'ca-certificates', 'xz-utils', 'pkg-config', 'time', 'libvulkan1', 'libvulkan-dev', 'vulkan-tools', 'glslc', 'libegl1', 'libglx0', 'libx11-6', 'libxext6', 'libx11-xcb1', 'libxcb1', 'libxcb-dri3-0', 'libxcb-present0', 'libxfixes3', 'libxrandr2', 'libdrm2', 'libexpat1', 'libwayland-client0'], returncode=0)

In [2]:
# 2) Cài Nim 2.2.10 chính chủ và xác minh checksum (tokenizer thuần Nim).
import hashlib, urllib.request, tarfile, os, subprocess
from pathlib import Path

NIM_VERSION = '2.2.10'
TOOLS = Path('/content/tools'); TOOLS.mkdir(exist_ok=True)
archive = TOOLS / f'nim-{NIM_VERSION}-linux_x64.tar.xz'
url = f'https://nim-lang.org/download/nim-{NIM_VERSION}-linux_x64.tar.xz'

# Tải file với User-Agent giả lập trình duyệt để tránh lỗi 403
if not archive.exists():
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(archive, 'wb') as out_file:
        out_file.write(response.read())

# Xác minh checksum
expected = '0a3a38752e97e9d44aa479b3a7b37336dfe0176daf22ee5b5218ad0991ecd211'
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual == expected, f'Nim checksum mismatch: {actual}'

# Giải nén
with tarfile.open(archive, 'r:xz') as tf:
    tf.extractall(TOOLS, filter='data')

# Cấu hình PATH
NIM_HOME = TOOLS / f'nim-{NIM_VERSION}'
os.environ['PATH'] = f"{NIM_HOME}/bin:" + os.environ['PATH']

# Kiểm tra phiên bản
subprocess.run(['nim', '--version'], check=True)


CompletedProcess(args=['nim', '--version'], returncode=0)

In [ ]:
# 3) Clone repository bản main mới nhất và tải submodule đã ghim trong repo.
import os, subprocess
from pathlib import Path

URL = 'https://github.com/bobbyshop-vui/nimformers-framework.git'
BRANCH = 'main'
REPO = Path('/content/nimformers-framework')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'sync', '--recursive'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'], check=True)
os.chdir(REPO)
print('Repo:', REPO, '| branch:', BRANCH)
print(subprocess.check_output(['git', 'log', '--oneline', '-1'], text=True))

In [4]:
# 4) Cài Vulkan + kiểm tra ICD NVIDIA và xác nhận dGPU (không chấp nhận lavapipe/CPU)
import subprocess, os, re
from pathlib import Path

# --- Bước 1: Cài vulkan-tools ---
print("[1/4] Cài vulkan-tools...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'vulkan-tools'], check=True)

# --- Bước 2: Xác định phiên bản driver NVIDIA và cài libnvidia-gl tương ứng ---
print("[2/4] Phát hiện phiên bản driver NVIDIA...")
smi = subprocess.check_output(['nvidia-smi'], text=True)
m = re.search(r'Driver Version:\s*(\d+)\.', smi)
assert m, "Không đọc được Driver Version từ nvidia-smi"
driver_major = m.group(1)
print(f"    -> Driver major: {driver_major}")

pkg = f'libnvidia-gl-{driver_major}'
print(f"[2/4] Cài {pkg}...")
subprocess.run(['apt-get', 'install', '-y', '-qq', pkg], check=True)

# --- Bước 3: Đảm bảo file ICD NVIDIA có thể được loader tìm thấy ---
print("[3/4] Cấu hình Vulkan ICD...")
Path('/usr/share/vulkan/icd.d').mkdir(parents=True, exist_ok=True)
candidates = [
    '/etc/vulkan/icd.d/nvidia_icd.json',
    '/usr/share/vulkan/icd.d/nvidia_icd.json',
    f'/usr/lib/x86_64-linux-gnu/libnvidia-gl-{driver_major}/nvidia_icd.json',
]
src = next((c for c in candidates if Path(c).exists()), None)
if src and src != '/usr/share/vulkan/icd.d/nvidia_icd.json':
    subprocess.run(['ln', '-sf', src, '/usr/share/vulkan/icd.d/nvidia_icd.json'], check=True)
    print(f"    -> symlink: {src} -> /usr/share/vulkan/icd.d/nvidia_icd.json")
else:
    print(f"    -> ICD có sẵn: {src}")

# --- Bước 4: Chạy kiểm tra ---
print("[4/4] Kiểm tra Vulkan...")
subprocess.run(['vulkaninfo', '--summary'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary = subprocess.check_output(['vulkaninfo', '--summary'],
                                   text=True, stderr=subprocess.STDOUT)
print(''.join(line for line in summary.splitlines()
              if any(k in line.lower() for k in ('device name', 'device type', 'driver name'))))

assert 'NVIDIA' in summary.upper(), \
    'Không thấy NVIDIA Vulkan ICD. Runtime GPU hiện tại không expose Vulkan graphics.'
assert 'DISCRETE_GPU' in summary.upper(), \
    'Không tìm thấy NVIDIA dGPU qua Vulkan; dừng để tránh fallback CPU/iGPU.'

print("✅ Vulkan OK: NVIDIA dGPU đã được expose.")

[1/4] Cài vulkan-tools...
[2/4] Phát hiện phiên bản driver NVIDIA...
    -> Driver major: 580
[2/4] Cài libnvidia-gl-580...
[3/4] Cấu hình Vulkan ICD...
    -> ICD có sẵn: /usr/share/vulkan/icd.d/nvidia_icd.json
[4/4] Kiểm tra Vulkan...

✅ Vulkan OK: NVIDIA dGPU đã được expose.


In [ ]:
# 5) Download the Codestral FP4-KM base, tokenizer and LoRA adapter. Hugging Face resumes interrupted downloads.
import subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download, model_info

MODELS = REPO / 'test/codestral/models'
HF_DIR = MODELS / 'hf'; HF_DIR.mkdir(parents=True, exist_ok=True)
BASE_REPO = 'bobbydeveloper2014/codestral-v0.1-7b-nimq'
BASE_REV = '148dd26885b5eeef03476683f05f0a4546cd75ba'
for name in ['codestral_mamba_only.nimq', 'config.json', 'generation_config.json', 'params.json',
             'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer.model.v3', 'tokenizer_config.json']:
    dest = HF_DIR if name.endswith('.json') or name.startswith('tokenizer.') else MODELS
    hf_hub_download(BASE_REPO, filename=name, revision=BASE_REV, local_dir=str(dest))
ADAPTER_REPO = 'bobbydeveloper2014/codestral-v0.1-nimq-qlora-linearattention'
ADAPTER_REV = model_info(ADAPTER_REPO).sha
adapter = hf_hub_download(ADAPTER_REPO, filename='jamba_parallel_lora.bin', revision=ADAPTER_REV, local_dir=str(MODELS))
Path(str(adapter) + '.hf_tokenizer_dir').write_text(str(HF_DIR.resolve()))
print('Adapter revision:', ADAPTER_REV)
for f in sorted(MODELS.iterdir()):
    if f.is_file():
        print(f'{f.name:40} {f.stat().st_size / 2**20:10.1f} MB')

In [ ]:
# 6) Compile the Vulkan shaders and the infer_lora binary.
import os
os.chdir(str(REPO))

!make shaders
!make build-infer-lora

## Inference

`codestral_mamba_only.nimq` được mmap thẳng từ đĩa. Trên T4, mỗi weight được copy một lần vào VRAM (`BYBY_VULKAN_DEVICE_LOCAL=auto`). Sau đó adapter LoRA được merge vào weight FP4 trên GPU. Log in thời gian merge của từng tensor.

Dòng `[vulkan] ... matrix=KHR_cooperative_matrix FP16/FP32 prefill=tensor-cores-only` xác nhận prefill chạy trên Tensor Core. Lần chạy đầu phải biên dịch shader bằng glslc nên chậm hơn; các lần sau dùng cache.

In [ ]:
# 7) Run inference (merged LoRA, greedy).
import os

PROMPT = 'You are a senior software engineer. Write a complete Python program to manage a library of books that meets these requirements: (1) Define a Book class with attributes: id, title, author, year, available (bool). (2) Define a Library class holding a list of books, with methods: add_book, remove_book(id), find_by_author(author), borrow_book(id), return_book(id), and list_available(). (3) Handle exceptions when borrowing an already-borrowed or nonexistent book, printing clear error messages. (4) Write a main() function demonstrating adding 5 books, borrowing/returning, searching by author, and listing available books. (5) Ensure full type hints, docstrings for every class and method, and PEP8 compliance. (6) Add unit tests using unittest for at least 4 methods, including error cases. (7) Briefly explain the time complexity of each method and suggest improvements if the list holds millions of books. Present the code in separate markdown blocks for the main code and the tests, with explanations in English.'
MAX_NEW_TOKENS = 999
model = MODELS / 'jamba_parallel_lora.bin'
base = MODELS / 'codestral_mamba_only.nimq'
assert model.exists() and base.exists() and (HF_DIR / 'tokenizer.model.v3').exists()
INFER_PROMPT = f'[INST] {PROMPT}[/INST]'

os.chdir(str(REPO))
print('Backend Vulkan | mmap | merged LoRA | max tokens =', MAX_NEW_TOKENS)

!BYBY_INFER_BACKEND=vulkan \
 BYBY_PROGRESS=0 \
 BYBY_MODEL_MMAP=1 \
 BYBY_VULKAN_DEVICE_LOCAL=auto \
 BYBY_VULKAN_TENSOR_CORES=1 \
 BYBY_MAX_NEW_TOKENS={MAX_NEW_TOKENS} \
 ./infer_lora_bin "{model}" "{INFER_PROMPT}" 0.0 "{HF_DIR}" "{base}"

## formers.n

Chạy cùng model bằng `formers-cli`. Nó tự nhận diện Mamba-2 và LoRA từ `hf/config.json`, header `.nimq` và `jamba_parallel_lora.bin`, merge LoRA trên GPU, rồi dùng nhánh checkpoint của `infer_lora`. Prompt thường được bọc thành `[INST] ...[/INST]`. Với `--verbose`, log có thêm các dòng `[vulkan] ... prefill=tensor-cores-only` và `[vulkan] mamba2 SSD prefill ...`. Phần text sinh ra là mốc để so với graph FIR ở cuối notebook.

In [ ]:
# 8) Build formers-cli and run Codestral with the merged LoRA through formers.n.
import os
os.chdir(str(REPO))
N_PREDICT = 200

!make formers-cli
!BYBY_MODEL_MMAP=1 \
 BYBY_VULKAN_DEVICE_LOCAL=auto \
 BYBY_VULKAN_TENSOR_CORES=1 \
 other/formers.n/formers-cli --models "{MODELS}" --backend vulkan -p "{PROMPT}" -n {N_PREDICT} --temp 0 --repeat-penalty 1.0 --no-progress --verbose

### Trace và chạy lifted FIR trên Tesla T4

Cell đầu chạy runtime Vulkan dưới `firlift --run` với LoRA đã merge, trên một prompt đúng 254 token. Như vậy mỗi kích thước prefill (128, 64, 32, 16, 8, 5 hàng) thành một phase, và bước decode cuối được lift thành `phase decode`. Trace đi qua pipe, không ghi file `.fprobe`. Thư mục output chỉ có `model.fir`, `config.json`, `trace.data` (khoảng 4 GB weight đã merge) và symlink tới adapter. Trên trace có `coopmatrix`, GEMM FP4 của prefill được lift thành micro-kernel Tensor Core `fp4_gemm_tensor_core_micro`.

Cell kế tiếp chạy graph đó trên cùng `PROMPT`, greedy, backend Vulkan. Token id được tạo bằng tokenizer native, nên khớp với cell formers.n ở trên. Tốc độ prefill và decode của graph được in riêng.

In [ ]:
# Lift: record the merged-LoRA Vulkan run on a 254-token prompt through a pipe, then write model.fir.
import os
import re
import subprocess
import tempfile
from collections import Counter
from pathlib import Path

os.chdir(str(REPO))

FIRLIFT = REPO / "other/formers.n/firlift/firlift"
FIRLIFT_SOURCE = REPO / "other/formers.n/firlift/firlift_main.nim"
FORMERS_CLI = REPO / "other/formers.n/formers-cli"
IDS_TOOL_SRC = Path("/content/tools/codestral_ids.nim")
IDS_TOOL = Path("/content/tools/codestral_ids")

FIR_OUT = Path("/content/codestral-t4-lifted")
FIR_OUT.mkdir(parents=True, exist_ok=True)
FIR_MODEL = FIR_OUT / "model.fir"
FIR_CONTEXT = 512
# 253 prompt rows = 128 + 64 + 32 + 16 + 8 + 5 (one phase each), then the decode step.
TRACE_TOKENS = 254
EXPECTED_PHASE_ROWS = {128, 64, 32, 16, 8, 5}

assert FORMERS_CLI.is_file(), f"formers-cli not found: {FORMERS_CLI}. Run the formers-cli cell first."
assert base.is_file() and model.is_file(), "Run the download cell first."


def run_streaming(command, env, what):
    """Run a command, stream combined output, and return it."""
    process = subprocess.Popen(
        [str(part) for part in command], cwd=str(REPO), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace",
    )
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    code = process.wait()
    if code:
        raise RuntimeError(f"{what} failed: exit code {code}")
    return "".join(lines)


# The native runtime's tokenizer (SentencePiece, BOS first), so the FIR prompt ids match it exactly.
IDS_TOOL_SRC.write_text("""## Codestral prompt ids with the native runtime's tokenizer.
import std/[os, strutils]
import hf_tokenizer

let tok = newHfTokenizer(paramStr(2))
case paramStr(1)
of "encode":
  echo "IDS ", encodeHf(tok, paramStr(3)).join(",")
of "decode":
  var ids: seq[int]
  for part in paramStr(3).splitWhitespace(): ids.add parseInt(part)
  echo "TEXT ", decodeHf(tok, ids)
else:
  quit "usage: codestral_ids encode|decode HF_DIR ARG"
""")
run_streaming(
    ["nim", "c", "-d:release", "--hints:off", f"--path:{REPO / 'test/codestral'}",
     f"--nimcache:{tempfile.mkdtemp(prefix='nimcache-codestral-ids-')}", f"-o:{IDS_TOOL}", IDS_TOOL_SRC],
    os.environ.copy(), "Building the tokenizer helper",
)


def codestral_ids(text):
    out = subprocess.run([str(IDS_TOOL), "encode", str(HF_DIR), text],
                         capture_output=True, text=True, check=True).stdout
    line = next(l for l in out.splitlines() if l.startswith("IDS "))
    return [int(x) for x in line[4:].split(",")]


def codestral_text(ids):
    out = subprocess.run([str(IDS_TOOL), "decode", str(HF_DIR), " ".join(map(str, ids))],
                         capture_output=True, text=True, check=True).stdout
    return out[out.index("TEXT ") + 5:]


def turn(text):
    return f"[INST] {text}[/INST]"


# Recording prompt: PROMPT plus one sentence, cut to <= 254 tokens, then padded one token at a time.
words = (PROMPT + " Finally, add a short README section that shows how to run the program "
         "and the tests from a shell.").split(" ")
k = len(words)
while k > 1 and len(codestral_ids(turn(" ".join(words[:k])))) > TRACE_TOKENS:
    k -= 1
trace_text = " ".join(words[:k])
while len(codestral_ids(turn(trace_text))) < TRACE_TOKENS:
    trace_text += " please"
TRACE_PROMPT = turn(trace_text)
assert len(codestral_ids(TRACE_PROMPT)) == TRACE_TOKENS
print("Trace prompt:", TRACE_TOKENS, "tokens")

# Rebuild firlift from this checkout.
run_streaming(
    ["nim", "c", "-d:release", "--hints:off", "--path:vendor/bybylang",
     f"--nimcache:{tempfile.mkdtemp(prefix='nimcache-firlift-t4-')}", f"-o:{FIRLIFT}", FIRLIFT_SOURCE],
    os.environ.copy(), "Building firlift",
)

lift_env = {k: v for k, v in os.environ.items() if not k.startswith(("FIRLIFT_", "BYBY_FIR_"))}
lift_env.update({
    "BYBY_MODEL_MMAP": "1",
    "BYBY_VULKAN_DEVICE_LOCAL": "auto",
    "BYBY_VULKAN_TENSOR_CORES": "1",
    "BYBY_MERGE_LORA": "1",
    "BYBY_MEMORY_LIMIT_MB": "10000",
    # State arenas and small tables are captured; checkpoint-sized uploads match --weights by fingerprint.
    "BYBY_FIR_PROBE_DATA_MAX": str(8 * 1024 * 1024),
})

lift_output = run_streaming(
    [
        FIRLIFT,
        "--config", HF_DIR / "config.json",
        "--context", FIR_CONTEXT,
        "--weights", base,
        "--weights", model,
        "--out", FIR_OUT,
        "--report",
        "--run",
        FORMERS_CLI,
        "--models", MODELS,
        "--backend", "vulkan",
        "-p", TRACE_PROMPT,
        "-n", "4",
        "--temp", "0",
        "--repeat-penalty", "1.0",
        "--no-progress",
        "--no-warmup",
    ],
    lift_env,
    "Recording and lifting the Vulkan run",
)

m = re.search(r"prompt eval\s*:\s*(\d+) tok", lift_output)
if not m or int(m.group(1)) != TRACE_TOKENS:
    raise RuntimeError(f"Runtime prompt tokens: {m.group(1) if m else '?'}; expected {TRACE_TOKENS}.")
assert FIR_MODEL.is_file(), f"Lifted graph missing: {FIR_MODEL}"

fir_lines = FIR_MODEL.read_text().splitlines()
phase_lines = [l.strip() for l in fir_lines if l.startswith("phase ")]
print("Lifted phases:")
for line in phase_lines:
    print(" ", line)
rows = {int(r) for r in re.findall(r"\brows=(\d+)", " ".join(phase_lines))}
if rows != EXPECTED_PHASE_ROWS:
    raise RuntimeError(f"Prefill phases {sorted(rows)}; expected {sorted(EXPECTED_PHASE_ROWS)}.")

fp4_nodes = Counter()
for line in fir_lines:
    words_ = line.split()
    if len(words_) >= 4 and words_[0] == "node" and "fp4_gemm" in words_[3]:
        fp4_nodes[words_[3]] += 1
print("FP4 GEMM nodes by kernel:", dict(fp4_nodes))
tensor_core = sum(n for name, n in fp4_nodes.items() if "tensor_core" in name)
if tensor_core == 0:
    print("WARNING: no fp4_gemm_tensor_core_micro node: the trace did not record a coopmatrix "
          "device, so prefill GEMMs were lifted as generic kernels.")
else:
    print("FP4 Tensor Core micro-kernel nodes:", tensor_core)

print(f"trace.data: {(FIR_OUT / 'trace.data').stat().st_size / 2**30:.2f} GiB")

check_env = os.environ.copy()
check_env["BYBY_VULKAN_TENSOR_CORES"] = "1"
run_streaming([FORMERS_CLI, "--fir-check", FIR_MODEL], check_env, "FIR graph check")

In [ ]:
# Run the lifted FIR graph on PROMPT (native tokenizer ids), greedy, with the decode step replayed each token.
import os
import subprocess

os.chdir(str(REPO))
FIR_TOKENS = 200
PROMPT_IDS = codestral_ids(turn(PROMPT))
assert len(PROMPT_IDS) + FIR_TOKENS - 1 <= FIR_CONTEXT, "prompt + tokens exceed the lifted context"
print("Prompt:", len(PROMPT_IDS), "tokens")

fir_env = os.environ.copy()
for key in ("BYBY_FIR_DECODE_REPLAY", "BYBY_FIR_DECODE_CB_NODES", "BYBY_VULKAN_COMMIT_WAIT",
            "BYBY_FIR_FP4_GEMV_TUNE", "BYBY_PREFILL_MIN_ROWS"):
    fir_env.pop(key, None)
fir_env.update({
    "BYBY_VULKAN_DEVICE_LOCAL": "auto",
    "BYBY_VULKAN_TENSOR_CORES": "1",
    "BYBY_FIR_PROFILE": "1",
})

process = subprocess.Popen(
    [str(part) for part in [
        FORMERS_CLI, "--models", MODELS, "--no-lora", "-m", base,
        "--config", FIR_OUT / "config.json", "--tokenizer", HF_DIR, "--fir", FIR_MODEL,
        "--prompt-ids", ",".join(map(str, PROMPT_IDS)), "-n", FIR_TOKENS,
        "--temp", "0", "--repeat-penalty", "1.0", "--backend", "vulkan",
        "--no-progress", "-c", FIR_CONTEXT,
    ]],
    cwd=str(REPO), env=fir_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, errors="replace",
)
lines = []
for line in process.stdout:
    print(line, end="", flush=True)
    lines.append(line)
if process.wait():
    raise RuntimeError("FIR run failed")

# With --prompt-ids the graph prints token ids; decode them with the native tokenizer.
id_lines = [l.split() for l in lines if l.strip() and all(w.isdigit() for w in l.split())]
if id_lines:
    generated = [int(w) for w in max(id_lines, key=len)]
    print("\n=== FIR text (compare with the formers.n cell) ===")
    print(codestral_text(generated))

### Tài liệu

- [Nim 2.2.10](https://nim-lang.org/download/)
- [Google Colab FAQ: GPU runtime](https://research.google.com/colaboratory/faq.html)
- [VK_KHR_cooperative_matrix](https://registry.khronos.org/vulkan/specs/latest/man/html/VK_KHR_cooperative_matrix.html)
- [Codestral NimQ trên Hugging Face](https://huggingface.co/bobbydeveloper2014/codestral-v0.1-7b-nimq)